# Bronze Layer — Ingestão dos Dados Brutos

Neste notebook lemos os oito arquivos enviados para o Volume e criamos uma tabela Delta Bronze correspondente para cada um deles. 

Conforme as boas práticas da camada Bronze, não aplicamos regras complexas de negócio aqui: as colunas são mantidas como texto/brutas, aplicamos o tratamento de codificação UTF-8/Windows-1252 onde necessário e adicionamos um metadado de auditoria (`ingestion_timestamp`). Utilizamos o modo `overwrite` para garantir total idempotência em caso de reexecuções pelo time.

In [0]:
%run ./00_Config

## Mapeamento e Ingestão Automatizada

Mapeamos os caminhos dos arquivos no Volume. Para os arquivos `Chamados_Hora.CSV` e `clientes.csv`, definimos explicitamente a codificação `windows-1252` para preservar acentuações e caracteres especiais em português.

In [0]:
from pyspark.sql.functions import current_timestamp

# Dicionário mapeando o nome da tabela Delta de destino, o arquivo de origem e opções específicas (como encoding)
data_sources = {
    "tb_base_atendentes": {"path": f"{landing_path}/base_atendentes.csv", "encoding": None},
    "tb_base_motivos": {"path": f"{landing_path}/base_motivos.csv", "encoding": None},
    "tb_canais": {"path": f"{landing_path}/canais.csv", "encoding": None},
    "tb_chamados": {"path": f"{landing_path}/chamados.csv", "encoding": None},
    "tb_chamados_hora": {"path": f"{landing_path}/Chamados_Hora.CSV", "encoding": "windows-1252"},
    "tb_clientes": {"path": f"{landing_path}/clientes.csv", "encoding": "windows-1252"},
    "tb_custos": {"path": f"{landing_path}/custos.csv", "encoding": None},
    "tb_pesquisa_satisfacao": {"path": f"{landing_path}/pesquisa_satisfacao.csv", "encoding": None}
}

# Dicionário auxiliar para capturar os DataFrames e exibir o sumário posteriormente
dfs_loaded = {}

for table_name, config_info in data_sources.items():
    file_path = config_info["path"]
    file_encoding = config_info["encoding"]
    
    # Leitura condicional aplicando o encoding quando necessário
    reader = spark.read.option("header", "true")
    if file_encoding:
        reader = reader.option("encoding", file_encoding)
        
    df_raw = reader.csv(file_path)
    dfs_loaded[table_name] = df_raw
    
    # Gravação idempotente com metadados de auditoria e atualização de schema
    df_raw \
        .withColumn("ingestion_timestamp", current_timestamp()) \
        .write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{bronze_schema}.{table_name}")

print("[SUCESSO] Ingestão da camada bronze finalizada de forma idempotente para todas as tabelas.")

## Validação da carga

Por fim, geramos um sumário com a contagem de registros para conferir o sucesso e o volume da ingestão em cada tabela.

In [0]:
load_summary = [
    ("base_atendentes.csv", dfs_loaded["tb_base_atendentes"].count()),
    ("base_motivos.csv", dfs_loaded["tb_base_motivos"].count()),
    ("canais.csv", dfs_loaded["tb_canais"].count()),
    ("chamados.csv", dfs_loaded["tb_chamados"].count()),
    ("Chamados_Hora.CSV", dfs_loaded["tb_chamados_hora"].count()),
    ("clientes.csv", dfs_loaded["tb_clientes"].count()),
    ("custos.csv", dfs_loaded["tb_custos"].count()),
    ("pesquisa_satisfacao.csv", dfs_loaded["tb_pesquisa_satisfacao"].count()),
]

df_load_summary = spark.createDataFrame(
    [
        ("base_atendentes.csv", dfs_loaded["tb_base_atendentes"].count()),
        ("base_motivos.csv", dfs_loaded["tb_base_motivos"].count()),
        ("canais.csv", dfs_loaded["tb_canais"].count()),
        ("chamados.csv", dfs_loaded["tb_chamados"].count()),
        ("Chamados_Hora.CSV", dfs_loaded["tb_chamados_hora"].count()),
        ("clientes.csv", dfs_loaded["tb_clientes"].count()),
        ("custos.csv", dfs_loaded["tb_custos"].count()),
        ("pesquisa_satisfacao.csv", dfs_loaded["tb_pesquisa_satisfacao"].count()),
    ],
    ["arquivo", "quantidade_registros"]
)

display(df_load_summary.orderBy("arquivo"))